# Contextual Subspace VQE with Generalized Transformations

This notebook demonstrates the complete workflow for:
1. Initializing molecular systems
2. Applying generalized fermionic transformations
3. Computing contextual subspace energies
4. Analyzing performance

In [2]:
%reload_ext autoreload
%autoreload 2

In [4]:
from symmerpyscf import (
    initialize_molecule,
    get_geometry,
    random_invertible_binary_matrix,
    mol_info_to_H_cs
)

## 1. Initialize Molecule

First, we initialize a molecular system. Here we use H2 with a bond length of 0.74 Å.

In [19]:
# Set parameters
bondlength = 0.74  # Angstroms
molecule = "LiH"
geometry = get_geometry(molecule=molecule, bondlength=bondlength)
geometry

[('Li', (0.0, 0.0, 0.0)), ('H', (0.0, 0.0, 0.74))]

In [60]:

basis = "sto-3g"
outdir = "./output"  # Optional: save data to file

# Initialize molecule and run quantum chemistry calculations
mol_info, pyscf_data, energy_data, filename = initialize_molecule(
    molecule=molecule,
    bondlength=bondlength,
    geometry=geometry,
    basis=basis,
    outdir=outdir,
    verbose=True
)

print(f"\nNumber of qubits: {pyscf_data['n_qubits']}")
print(f"Number of electrons: {pyscf_data['n_particles']['total']}")
print(f"Point group: {pyscf_data['point_group']['groupname']}")

Hartree-Fock energy for H1-Li1_sto-3g_singlet (4 electrons) is -7.543569918023586
MP2 energy for H1-Li1_sto-3g_singlet (4 electrons) is -7.556157508927201
CCSD energy for H1-Li1_sto-3g_singlet (4 electrons) is -7.562398014850482
CISD energy for H1-Li1_sto-3g_singlet (4 electrons) is -7.562392190378417
FCI energy for H1-Li1_sto-3g_singlet (4 electrons) is -7.562405906832996

Number of qubits: 12
Number of electrons: 4
Point group: Coov


### Print Energy Summary

## 2. Apply Generalized Transformation

We can use different fermionic-to-qubit mappings by specifying the beta matrix:
- `'Jordan-Wigner'`: Standard Jordan-Wigner transformation (identity matrix)
- `'Bravyi-Kitaev'`: Bravyi-Kitaev transformation
- Random invertible matrix: For exploring alternative mappings

In [21]:
# Option 1: Jordan-Wigner transformation
beta_jw = random_invertible_binary_matrix(
    n=pyscf_data['n_qubits'],
    beta='Jordan-Wigner'
)

print("Jordan-Wigner beta matrix:")
print(beta_jw)

# Option 2: Bravyi-Kitaev transformation
beta_bk = random_invertible_binary_matrix(
    n=pyscf_data['n_qubits'],
    beta='Bravyi-Kitaev'
)

print("\nBravyi-Kitaev beta matrix:")
print(beta_bk)

# Option 3: Random invertible matrix
beta_random = random_invertible_binary_matrix(
    n=pyscf_data['n_qubits']
)

print("\nRandom beta matrix:")
print(beta_random)

Jordan-Wigner beta matrix:
[[1 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1]]

Bravyi-Kitaev beta matrix:
[[1 1 1 1 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 1]]

Random beta matrix:
[[1 0 0 0 0 1 1 1 1 1 0 0]
 [0 1 1 1 0 1 1 0 0 1 0 0]
 [0 0 0 1 1 1 0 0 1 1 1 1]
 [1 1 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 1 0 0]
 [1 1 1 0 0 0 0 0 0 0 1 0]
 [1 0 1 0 0 0 0 0 1 1 1 1]
 [1 1 0 1 1 0 0 1 0 1 0 1]
 [0 0 0 0 1 1 1 0 1 1 1 1]
 [1 1 1 0 0 1 0 0 1 1 1 0]
 [1 

## 3. Contextual Subspace Calculation

Now we compute the energy in a contextual subspace with reduced qubit count.

In [29]:
# Set contextual subspace parameters
n_cs_qubit = 6  # Target number of qubits

# Use Jordan-Wigner transformation
beta = beta_jw

# Compute baseline NCS energy (1 qubit contextual subspace)
data_ncs = mol_info_to_H_cs(
    mol_info,
    n_cs_qubit=1,
    beta=beta
)

ncs_energy = data_ncs['cs_energy']
fci_energy = data_ncs['fci_energy']

print(f"NCS Energy (1 qubit): {ncs_energy:.8f} Ha")
print(f"FCI Energy: {fci_energy:.8f} Ha")
print(f"Error vs FCI: {ncs_energy - fci_energy:.8e} Ha\n")

# Compute with target number of qubits
data_cs = mol_info_to_H_cs(
    mol_info,
    n_cs_qubit=n_cs_qubit,
    beta=beta
)

print(f"CS Energy ({n_cs_qubit} qubits): {data_cs['cs_energy']:.8f} Ha")
print(f"Error vs FCI: {data_cs['cs_energy'] - fci_energy:.8e} Ha")
print(f"Number of Hamiltonian terms: {data_cs['n_terms_hamiltonian']}")
print(f"Number of CCSD generator terms: {data_cs['n_terms_ccsd_generator']}")

NCS Energy (1 qubit): -7.54356992 Ha
FCI Energy: -7.56240591 Ha
Error vs FCI: 1.88359888e-02 Ha

CS Energy (6 qubits): -7.56153913 Ha
Error vs FCI: 8.66778814e-04 Ha
Number of Hamiltonian terms: 193
Number of CCSD generator terms: 84


## 4. Compare Different Transformations

In [58]:
transformations = {
    'Jordan-Wigner': beta_jw,
    'Bravyi-Kitaev': beta_bk,
    'Random': beta_random
}

results = {}
datas = []
for name, beta_matrix in transformations.items():
    data = mol_info_to_H_cs(
        mol_info,
        n_cs_qubit=n_cs_qubit,
        beta=beta_matrix
    )
    results[name] = data
    datas.append(data)

    print(f"\n{name} Transformation:")
    print(f"  CS Energy: {data['cs_energy']:.8f} Ha")
    print(f"  Error: {data['cs_energy'] - fci_energy:.8e} Ha")
    print(f"  Hamiltonian terms: {data['n_terms_hamiltonian']}")
    print(f"  HF contextual state {data['hf_cs']}")


Jordan-Wigner Transformation:
  CS Energy: -7.56153913 Ha
  Error: 8.66778814e-04 Ha
  Hamiltonian terms: 193
  HF contextual state  1.000 |100010>

Bravyi-Kitaev Transformation:
  CS Energy: -7.56153913 Ha
  Error: 8.66778814e-04 Ha
  Hamiltonian terms: 193
  HF contextual state  1.000 |000011>

Random Transformation:
  CS Energy: -7.56153913 Ha
  Error: 8.66778814e-04 Ha
  Hamiltonian terms: 193
  HF contextual state  1.000 |101111>


## 5. Test loading molecular json file compatible with Symmer demo notebook

The molecular data can be saved to JSON format for later use.

In [73]:
import json

# Load saved data (if outdir was specified)
if filename:
    with open(filename, 'r') as f:
        chem_data = json.load(f)
    print('Loaded data from:',filename)
    print("Loaded data keys:")
    print(list(chem_data.keys()))

    print('\nSubkey under ["calculated_properties"]\n')
    print(list(chem_data['calculated_properties'].keys()))
    print('\nSubkey under ["auxiliary_operators"]\n')
    print(list(chem_data['auxiliary_operators'].keys()))

    
    # Access specific data
    print(f"\nBasis set: {chem_data['basis']}")
    print(f"Number of qubits: {chem_data['n_qubits']}")
    print(f"FCI energy: {chem_data['calculated_properties']['FCI']['energy']:.8f} Ha")

Loaded data from: ./output/LiH_0.740_sto-3g.json
Loaded data keys:
['H', 'H_second_quantized', 'qubit_encoding', 'unit', 'geometry', 'basis', 'charge', 'spin', 'hf_array', 'hf_state', 'hf_method', 'n_particles', 'n_qubits', 'convergence_threshold', 'point_group', 'calculated_properties', 'auxiliary_operators']

Subkey under ["calculated_properties"]

['HF', 'MP2', 'CISD', 'CCSD', 'FCI']

Subkey under ["auxiliary_operators"]

['number_operator', 'N_alpha', 'N_beta', 'S^2_operator', 'CCSD_operator', 'number_operator_second_quantized', 'N_alpha_second_quantized', 'N_beta_second_quantized', 'S^2_operator_second_quantized', 'CCSD_operator_second_quantized', 'ccsd_state', 'cisd_state', 'fci_state']

Basis set: sto-3g
Number of qubits: 12
FCI energy: -7.56240591 Ha


In [68]:
from symmer import PauliwordOp, QuantumState

H = PauliwordOp.from_dictionary(chem_data['H'])

hf_state = QuantumState.from_dictionary(chem_data['hf_state'])
cisd_state = QuantumState.from_dictionary(chem_data['auxiliary_operators']['cisd_state'])
ccsd_state = QuantumState.from_dictionary(chem_data['auxiliary_operators']['ccsd_state'])
fci_state = QuantumState.from_dictionary(chem_data['auxiliary_operators']['fci_state'])

number_alpha = PauliwordOp.from_dictionary(chem_data['auxiliary_operators']['N_alpha'])
number_beta  = PauliwordOp.from_dictionary(chem_data['auxiliary_operators']['N_beta'])
S2_op = PauliwordOp.from_dictionary(chem_data['auxiliary_operators']['S^2_operator'] )
fci_energy = chem_data['calculated_properties']['FCI']['energy']


In [70]:
H.expval(fci_state) - fci_energy

np.complex128(1.5987211554602254e-14+0j)